# The search: the trap in us

*Notebook journey for §5 of the dissertation. This chapter's trap is not in any dataset; it is in our own searching. To measure it cleanly we reuse real features but scramble the labels into a coin, so the truth is exactly 0.5, then watch our ordinary deep-learning habits invent a score anyway.*

## a. The question, and a clean test

In [1]:
import numpy as np, os

# reuse the LOAN features (real numbers about real people), but replace the labels with a
# coin flip, unrelated to the features. Now there is nothing to learn: the truth is exactly 0.5.
DATA = "data/loan_uci350.csv" if os.path.exists("data/loan_uci350.csv") else "../data/loan_uci350.csv"
A = np.loadtxt(DATA, delimiter=",", skiprows=1)
X = A[:, :-1]
rng = np.random.default_rng(0)
y = rng.integers(0, 2, size=len(X))          # a fair coin per client, nothing to do with X

print("clients:", len(X), "| features:", X.shape[1])
print("label balance:", round(float(y.mean()), 3), "(a fair coin)")
print("corr(feature 0, label):", round(float(np.corrcoef(X[:, 0], y)[0, 1]), 4), "(~0: nothing to learn)")

# split: a training set, a SMALL validation set (the fuel for snooping), a LARGE sealed test (the truth)
perm = rng.permutation(len(X))
tr, va, te = perm[:4000], perm[4000:4200], perm[4200:14200]
print("\ntrain", len(tr), "| validation", len(va), "(small: the fuel)", "| sealed test", len(te), "(large: the truth)")
print("truth by construction: no model can beat 0.5 on this data")

clients: 30000 | features: 23
label balance: 0.499 (a fair coin)
corr(feature 0, label): -0.0022 (~0: nothing to learn)

train 4000 | validation 200 (small: the fuel) | sealed test 10000 (large: the truth)
truth by construction: no model can beat 0.5 on this data


## b. Early stopping is a search

We train one model, nothing fancy, and let it run for many epochs. At each epoch we peek at the validation score and remember the best one, which is exactly what early stopping does. We also secretly check the sealed test each epoch, to see the truth the validation is hiding.

In [2]:
# the model (same small network from Section 1)
def init(d, width, K, rng):
    return [rng.standard_normal((d, width)) * 0.1, np.zeros(width),
            rng.standard_normal((width, K)) * 0.1, np.zeros(K)]
def forward(Xb, p):
    W1, b1, W2, b2 = p
    a = Xb @ W1 + b1; h = np.maximum(a, 0.0); z = h @ W2 + b2
    return a, h, z
def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def accuracy(p, Xb, yb):
    return float((forward(Xb, p)[2].argmax(1) == yb).mean())

# standardize on the training side only, then keep val and sealed test fixed
m, sd = X[tr].mean(0), X[tr].std(0) + 1e-9
Xtr, Xva, Xte = (X[tr] - m) / sd, (X[va] - m) / sd, (X[te] - m) / sd
ytr, yva, yte = y[tr], y[va], y[te]

# train ONE model for many epochs; at every epoch record BOTH the validation and the sealed-test score
def train_track(width, lr, epochs, seed):
    rng = np.random.default_rng(seed)
    p = init(Xtr.shape[1], width, 2, rng); n = len(ytr)
    Y = np.zeros((n, 2)); Y[np.arange(n), ytr] = 1.0
    hist = []
    for e in range(epochs):
        a, h, z = forward(Xtr, p); dz = (softmax(z) - Y) / n
        W1, b1, W2, b2 = p
        dW2 = h.T @ dz; db2 = dz.sum(0); da = (dz @ W2.T) * (a > 0)
        dW1 = Xtr.T @ da; db1 = da.sum(0)
        p = [W1 - lr*dW1, b1 - lr*db1, W2 - lr*dW2, b2 - lr*db2]
        hist.append((e, accuracy(p, Xva, yva), accuracy(p, Xte, yte)))
    return hist

hist = train_track(width=32, lr=0.3, epochs=300, seed=1)
best = max(hist, key=lambda r: r[1])              # early stopping keeps the best-validation epoch
print("one model, 300 epochs, on data with nothing to learn.")
print("validation wobbles by luck across the epochs (a sample):")
for e, v, t in hist[::50]:
    print(f"  epoch {e:>3}: validation {v:.3f}   sealed test {t:.3f}")
print(f"\nearly stopping keeps the luckiest epoch (epoch {best[0]}):")
print(f"  validation  {best[1]:.3f}   <- what we would proudly report")
print(f"  sealed test {best[2]:.3f}   <- the truth (still a coin)")
print(f"  the gap, pure luck from peeking every epoch: +{best[1] - best[2]:.3f}")

one model, 300 epochs, on data with nothing to learn.
validation wobbles by luck across the epochs (a sample):
  epoch   0: validation 0.530   sealed test 0.494
  epoch  50: validation 0.495   sealed test 0.487
  epoch 100: validation 0.505   sealed test 0.494
  epoch 150: validation 0.525   sealed test 0.498
  epoch 200: validation 0.540   sealed test 0.500
  epoch 250: validation 0.495   sealed test 0.499

early stopping keeps the luckiest epoch (epoch 164):
  validation  0.545   <- what we would proudly report
  sealed test 0.500   <- the truth (still a coin)
  the gap, pure luck from peeking every epoch: +0.045


## c. Seeds pile on, and the name for it

One early-stopped run already peeked three hundred times. Now the other ordinary habit: run the same setup under a few different random seeds and keep the best. Each seed is a fresh handful of luck; keeping the best of several stacks more searching on top of the searching we already did.

In [3]:
# one run already early-stops (200 epochs). Now the OTHER habit: try a few seeds, keep the best validation.
def early_stop_run(seed, width=32, lr=0.3, epochs=200):
    rng = np.random.default_rng(seed)
    p = init(Xtr.shape[1], width, 2, rng); n = len(ytr)
    Y = np.zeros((n, 2)); Y[np.arange(n), ytr] = 1.0
    best_v, best_p = -1.0, None
    for e in range(epochs):
        a, h, z = forward(Xtr, p); dz = (softmax(z) - Y) / n
        W1, b1, W2, b2 = p
        dW2 = h.T @ dz; db2 = dz.sum(0); da = (dz @ W2.T) * (a > 0)
        dW1 = Xtr.T @ da; db1 = da.sum(0)
        p = [W1 - lr*dW1, b1 - lr*db1, W2 - lr*dW2, b2 - lr*db2]
        v = accuracy(p, Xva, yva)                 # peek validation every epoch (early stopping)
        if v > best_v:
            best_v, best_p = v, [w.copy() for w in p]
    return best_v, accuracy(best_p, Xte, yte)     # open the sealed test once, on the kept model

def sweep(N, reps=12):
    ks, ts = [], []
    for r in range(reps):
        runs = [early_stop_run(1000 * r + s) for s in range(N)]
        bv, bt = max(runs, key=lambda x: x[0])    # keep the best VALIDATION across the N seeds
        ks.append(bv); ts.append(bt)
    return float(np.mean(ks)), float(np.mean(ts))

print("try N seeds (each already early-stopped over 200 epochs), keep the best validation:")
for N in (1, 5, 20):
    k, t = sweep(N)
    print(f"  {N:>2} seeds: kept validation {k:.3f}   sealed test {t:.3f}   (really {N} x 200 = {N*200} hidden tries)")

try N seeds (each already early-stopped over 200 epochs), keep the best validation:


   1 seeds: kept validation 0.545   sealed test 0.496   (really 1 x 200 = 200 hidden tries)


   5 seeds: kept validation 0.581   sealed test 0.496   (really 5 x 200 = 1000 hidden tries)


  20 seeds: kept validation 0.607   sealed test 0.497   (really 20 x 200 = 4000 hidden tries)


## d. But search is how we improve

Searching is not the villain. On data with a real signal, the same searching that invented a score out of noise can genuinely find a better model, as long as we search in the open and keep the test sealed. Here we compare two architectures and two optimisers on the real loan labels, pick the best on the validation set, and only then open the sealed test.

In [4]:
# But search is not the villain. On the REAL loan labels (real signal), honest search IMPROVES the model.
yr = A[:, -1].astype(int)                          # the real default labels, not the coin

def init_net(d, widths, K, rng):
    sizes = [d] + list(widths) + [K]
    return [[rng.standard_normal((sizes[i], sizes[i + 1])) * 0.1, np.zeros(sizes[i + 1])] for i in range(len(sizes) - 1)]

def forward_net(Xb, net):
    acts, pre, h = [Xb], [], Xb
    for i, (W, b) in enumerate(net):
        z = h @ W + b; pre.append(z)
        h = np.maximum(z, 0.0) if i < len(net) - 1 else z
        acts.append(h)
    return acts, pre

def train_net(Xtr, ytr, widths, lr, epochs, opt, rng, beta=0.9):
    net = init_net(Xtr.shape[1], widths, 2, rng)
    n = len(ytr); Y = np.zeros((n, 2)); Y[np.arange(n), ytr] = 1.0
    vel = [[np.zeros_like(W), np.zeros_like(b)] for W, b in net]
    for e in range(epochs):
        acts, pre = forward_net(Xtr, net)
        dz = (softmax(acts[-1]) - Y) / n
        grads = [None] * len(net)
        for i in reversed(range(len(net))):
            grads[i] = [acts[i].T @ dz, dz.sum(0)]
            if i > 0:
                dz = (dz @ net[i][0].T) * (pre[i - 1] > 0)     # back through the previous ReLU
        for i in range(len(net)):
            dW, db = grads[i]
            if opt == "momentum":
                vel[i][0] = beta * vel[i][0] + dW; vel[i][1] = beta * vel[i][1] + db
                net[i][0] -= lr * vel[i][0]; net[i][1] -= lr * vel[i][1]
            else:
                net[i][0] -= lr * dW; net[i][1] -= lr * db
    return net

def bal(net, Xb, yb):
    yhat = forward_net(Xb, net)[0][-1].argmax(1)
    rec = ((yhat == 1) & (yb == 1)).sum() / max((yb == 1).sum(), 1)
    spec = ((yhat == 0) & (yb == 0)).sum() / max((yb == 0).sum(), 1)
    return float((rec + spec) / 2)

# a fresh honest split: train / validation / a sealed test we open only at the end
rng2 = np.random.default_rng(7)
perm = rng2.permutation(len(X))
trr, var_, ter = perm[:15000], perm[15000:18000], perm[18000:]
mm, ss = X[trr].mean(0), X[trr].std(0) + 1e-9
Xtr2, Xva2, Xte2 = (X[trr] - mm) / ss, (X[var_] - mm) / ss, (X[ter] - mm) / ss

print("compare 2 architectures x 2 optimisers, honestly (balanced accuracy):")
print(f"  {'architecture':<17}{'optimiser':<11}{'validation':<12}sealed test")
results = []
for widths, name in [([16], "1 hidden layer"), ([32, 16], "2 hidden layers")]:
    for opt, o in (("plain GD", "gd"), ("momentum", "momentum")):
        net = train_net(Xtr2, yr[trr], widths, 0.1, 300, o, np.random.default_rng(0))
        v, t = bal(net, Xva2, yr[var_]), bal(net, Xte2, yr[ter])
        results.append((name, opt, v, t))
        print(f"  {name:<17}{opt:<11}{v:<12.3f}{t:.3f}")

best = max(results, key=lambda r: r[2])            # pick the best on VALIDATION, then open the test
print(f"\nwe keep the best on validation: {best[0]}, {best[1]}")
print(f"  its sealed-test balanced accuracy: {best[3]:.3f}   (the single default from Section 2 was 0.644)")

compare 2 architectures x 2 optimisers, honestly (balanced accuracy):
  architecture     optimiser  validation  sealed test


  1 hidden layer   plain GD   0.624       0.624


  1 hidden layer   momentum   0.660       0.650


  2 hidden layers  plain GD   0.618       0.621


  2 hidden layers  momentum   0.658       0.653

we keep the best on validation: 1 hidden layer, momentum
  its sealed-test balanced accuracy: 0.650   (the single default from Section 2 was 0.644)
